# Structured Data Models - TabICL Quickstart

**Structured Data Models (SDM)** is a modular library for in-context foundation models on structured data. It keeps table data in typed tensor containers and runs predictions from labeled context rows without per-task training.

This notebook uses three core pieces: a lossless `TableTensor`, a pretrained `TabICLv2` model, and a configurable `Recipe` for pre- and post-processing.


## 0. Installation

In Colab, the next cell installs `structured-data-models` from GitHub when the package is not already available. In a local checkout where the package has already been installed, it is a no-op.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sdm") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/NVIDIA/structured-data-models.git",
        ]
    )

In [ ]:
import time

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_dtype = torch.bfloat16
if device.type == "cuda" and torch.cuda.get_device_capability(0)[0] < 8:
    amp_dtype = torch.float16

## 1. Load a pretrained model and predict

Wrap a dataframe in a `TableTensor`, split it into context and query rows, and call the model.


In [ ]:
from sdm import TableTensor, infer_stypes
from sdm.models import TabICLv2
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error, r2_score

df = load_diabetes(as_frame=True).frame.sample(frac=1, random_state=0)
table = TableTensor.from_pandas(df=df, stypes=infer_stypes(df), device=device)

n_context = 300
x = table.drop_columns("target")
y = table[:, "target"]
x_ctx = x[:n_context]
y_ctx = y[:n_context]
x_qry = x[n_context:]
y_true = df["target"].to_numpy()[n_context:]

model = TabICLv2(device=device)
generator = torch.Generator(device=device).manual_seed(0)
with (
    torch.inference_mode(),
    torch.amp.autocast(
        device.type,
        amp_dtype,
        enabled=table.is_cuda,
    ),
):
    out = model(
        x_context=x_ctx,
        y_context=y_ctx,
        x_query=x_qry,
        num_estimators=2,
        generator=generator,
    )

pred = out.numerical.float().cpu().numpy().mean(axis=-1).reshape(-1)
rmse = mean_squared_error(y_true, pred) ** 0.5
print(f"RMSE={rmse:.4f}  R2={r2_score(y_true, pred):.4f}")

## 2. Represent data on the accelerator

A `TableTensor` keeps each semantic column type in its own block on the selected device.


In [ ]:
df.head(3)

In [ ]:
stypes = infer_stypes(df)

In [ ]:
stypes

In [ ]:
table = TableTensor.from_pandas(df=df, stypes=stypes, device=device)
table[:3]

## 3. Inspect and customize pre/post processing

TabICLv2 ships a `default_recipe()`: the exact pre/post-processing it expects, as an inspectable, editable object.


In [ ]:
recipe = TabICLv2.default_recipe()
recipe

The default numerical normalization can be swapped for another processor. This compares the default recipe with a recipe that forces a normal-output quantile transform.


In [ ]:
from sdm.processing import (
    Clip,
    ClipSigma,
    DropConstantColumns,
    ImputeMean,
    QuantileTransform,
    Recipe,
    ReduceEstimators,
    ShuffleColumns,
    Standardize,
)

generator = torch.Generator(device=device).manual_seed(0)
with (
    torch.inference_mode(),
    torch.amp.autocast(
        device.type,
        amp_dtype,
        enabled=table.is_cuda,
    ),
):
    out = model(
        x_context=x_ctx,
        y_context=y_ctx,
        x_query=x_qry,
        num_estimators=2,
        generator=generator,
    )
    pred = out.numerical.float().cpu().numpy().mean(axis=-1).reshape(-1)
    rmse_default = mean_squared_error(y_true, pred) ** 0.5

custom_recipe = Recipe(
    features=[
        ImputeMean(),
        DropConstantColumns(),
        Standardize(epsilon=1e-6),
        Clip(min_value=-100.0, max_value=100.0),
        QuantileTransform(output_distribution="normal"),
        ClipSigma(threshold=4.0),
        ShuffleColumns(method="shift"),
    ],
    target=Standardize(),
    output=ReduceEstimators(method="mean"),
)

generator = torch.Generator(device=device).manual_seed(0)
with (
    torch.inference_mode(),
    torch.amp.autocast(
        device.type,
        amp_dtype,
        enabled=table.is_cuda,
    ),
):
    out = model(
        x_context=x_ctx,
        y_context=y_ctx,
        x_query=x_qry,
        num_estimators=2,
        recipe=custom_recipe,
        generator=generator,
    )
    pred = out.numerical.float().cpu().numpy().mean(axis=-1).reshape(-1)
    rmse_quantile = mean_squared_error(y_true, pred) ** 0.5

print(f"default recipe: RMSE={rmse_default:.4f}")
print(f"quantile recipe: RMSE={rmse_quantile:.4f}")

## 4. Fast repeated inference with key/value caching

Encode the context once with `fit()`, then answer many query batches cheaply with `predict()`.


In [ ]:
def timed(fn, warmup=1, iters=3):
    for _ in range(warmup):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters


def predict():
    model(x_context=x_ctx, y_context=y_ctx, x_query=x_qry, num_estimators=2)


def predict_with_cache():
    model.predict(x_qry)


with (
    torch.inference_mode(),
    torch.amp.autocast(
        device.type,
        amp_dtype,
        enabled=table.is_cuda,
    ),
):
    t_nocache = timed(predict)
    model.fit(x=x_ctx, y=y_ctx, num_estimators=2)
    t_cache = timed(predict_with_cache)
model.clear()

print(f"no cache: {t_nocache * 1e3:6.1f} ms")
print(f"cache:    {t_cache * 1e3:6.1f} ms")
print(f"speedup:  {t_nocache / t_cache:.1f}x")

## 5. Ablate ensembling with `num_estimators`

Each ensemble member sees an independently sampled view; averaging members reduces variance and restores invariances the model was trained with.


In [ ]:
for n_estimators in [1, 2, 8]:
    generator = torch.Generator(device=device).manual_seed(0)
    with (
        torch.inference_mode(),
        torch.amp.autocast(
            device.type,
            amp_dtype,
            enabled=table.is_cuda,
        ),
    ):
        out = model(
            x_context=x_ctx,
            y_context=y_ctx,
            x_query=x_qry,
            num_estimators=n_estimators,
            generator=generator,
        )
    pred = out.numerical.float().cpu().numpy().mean(axis=-1).reshape(-1)
    rmse = mean_squared_error(y_true, pred) ** 0.5
    print(f"num_estimators={n_estimators:>2}  RMSE={rmse:.4f}")

## 6. Same interface, different task

The identical `model(x_context, y_context, x_query)` call handles classification.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

bc_df = load_breast_cancer(as_frame=True).frame
context_df, query_df = train_test_split(
    bc_df,
    train_size=400,
    random_state=0,
    stratify=bc_df["target"],
)
bc_table = TableTensor.from_pandas(
    df=pd.concat([context_df, query_df]),
    stypes=infer_stypes(bc_df, overrides={"target": "categorical"}),
    device=device,
)

x_bc = bc_table.drop_columns("target")
y_bc = bc_table[:, "target"]
generator = torch.Generator(device=device).manual_seed(0)
with (
    torch.inference_mode(),
    torch.amp.autocast(
        device.type,
        amp_dtype,
        enabled=bc_table.is_cuda,
    ),
):
    out = model(
        x_context=x_bc[:400],
        y_context=y_bc[:400],
        x_query=x_bc[400:],
        num_estimators=2,
        generator=generator,
    )

p1 = out[:, "1"].numerical.float().cpu().numpy().reshape(-1)
y_true = query_df["target"].to_numpy()
auc = roc_auc_score(y_true, p1)
acc = accuracy_score(y_true, (p1 > 0.5).astype(np.int64))
print(f"ROC-AUC={auc:.4f}  accuracy={acc:.4f}")